In [59]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, mean_squared_error
clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")
display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


## Experiment A : Classification

In [60]:
numeric_features = [
    "complexity_score",
    "team_experience_years",
    "customer_tenure_months"
]

categorical_features = [
    "priority",
    "channel"
]

print("Numerical features of clf:", numeric_features)
print("Categorical features of clf:", categorical_features)


X_clf = clf_df.drop(columns=["ticket_id", "escalated"])
y_clf = clf_df["escalated"]

# Stratified train/test split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.25,
    random_state=42,
    stratify=y_clf   # stratification helps to make the distribution of escalated balanced across the train/test split
)

print("Training features:", X_train_clf.shape)
print("Testing features:", X_test_clf.shape)

print("\nTraining target distribution:")
print(y_train_clf.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test_clf.value_counts(normalize=True))


Numerical features of clf: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features of clf: ['priority', 'channel']
Training features: (675, 5)
Testing features: (225, 5)

Training target distribution:
escalated
0    0.788148
1    0.211852
Name: proportion, dtype: float64

Testing target distribution:
escalated
0    0.786667
1    0.213333
Name: proportion, dtype: float64


In [61]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_clf = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

Here numeric_transformer is used for imputation and scaling of the numeric_features, I used median for imputation.
categorical_transformer used for the categorical_features, I used most_frequent/mode for imputation and changed it to numeric value using one-hot encoding.

In [62]:

models_clf = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    )
}
#cross validation
cv_clf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

classification_results = []

for name, model in models_clf.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_clf),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train_clf,
        y_train_clf,
        cv=cv_clf,
        scoring="f1",
        return_train_score=False
    )

    classification_results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    })

classification_results_df = pd.DataFrame(classification_results)

display(
    classification_results_df.sort_values(
        by="Mean F1",
        ascending=False
    )
)


,Model,Mean F1,Std F1
2,Decision Tree,0.401449,0.104140
0,Logistic Regression,0.382200,0.053838
1,KNN,0.309444,0.073750


We will tune the logistic regression, eventhough the decision tree got the highest mean F1 the logistic regression has the lowest std of F1, which means less deviations and more reliable.

In [63]:

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_clf),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

param_grid_clf = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid_clf = GridSearchCV(
    logistic_pipeline,
    param_grid_clf,
    cv=cv_clf,
    scoring="f1",
    n_jobs=-1
)

grid_clf.fit(X_train_clf, y_train_clf)

print("Best parameters:", grid_clf.best_params_)
print("Best CV F1:", grid_clf.best_score_)


Best parameters: {'model__C': 10}
Best CV F1: 0.3949600999465296


I used GridSearchCV to find the best regularization strength (C) for our Logistic Regression model. The pipeline combines preprocessing and classification.

Different C values: 0.01, 0.1, 1, 10 and 100, tested using 5-fold cross-validation and F1-score as the evaluation metric. GridSearchCV selects the value that achieves the highest average F1-score. Prints the best hyperparameter and its corresponding cross-validation F1-score. This helps us identify a model configuration that balances predictive performance and overfitting.

The final F1 result is improved to 0.3949...

### Final Test Evaluation

In [74]:

best_clf = grid_clf.best_estimator_

y_pred_clf = best_clf.predict(X_test_clf)

print("Confusion Matrix:")
print(confusion_matrix(y_test_clf, y_pred_clf))

print("\nClassification Report:")
print(classification_report(y_test_clf, y_pred_clf))


Confusion Matrix:
[[169   8]
 [ 38  10]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.95      0.88       177
           1       0.56      0.21      0.30        48

    accuracy                           0.80       225
   macro avg       0.69      0.58      0.59       225
weighted avg       0.76      0.80      0.76       225



We evaluate the best Logistic Regression pipeline on the held-out test set to measure its performance on unseen data. The confusion matrix summarizes correct and incorrect predictions, while the classification report provides precision, recall, F1-score and accuracy. We use these results to assess the model's ability to identify escalated tickets and examine false positives and false negatives.

- false positive is when the model predictes escalated but the reality is not escalated
- false negative is when the model predictes not escalated but in reality it is escalated

## Experiment B : Regression

In [64]:
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
y_reg = reg_df["resolution_time_hours"]

print("Features shape:", X_reg.shape)
print("Target shape:", y_reg.shape)

Features shape: (800, 5)
Target shape: (800,)


In [65]:
numeric_features_reg = [
    "complexity_score",
    "team_experience_years",
    "customer_tenure_months"
]

categorical_features_reg = [
    "priority",
    "channel"
]

print("Numerical features of reg:", numeric_features_reg)
print("Categorical features of reg:", categorical_features_reg)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=42,
)

print("Training features:", X_train_reg.shape)
print("Testing features:", X_test_reg.shape)

Numerical features of reg: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features of reg: ['priority', 'channel']
Training features: (600, 5)
Testing features: (200, 5)


In [66]:
numeric_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_reg = ColumnTransformer(transformers=[
    ("num", numeric_transformer_reg, numeric_features_reg),
    ("cat", categorical_transformer_reg, categorical_features_reg)
])

This is all the same as the previous one, numeric values are imputed using median and then scaled. the categorical values are imputed using  mode and changed to numeric values using one-hot encoding.

In [67]:
models_reg = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree Regressor": DecisionTreeRegressor(
        random_state=42
    )
}

- Linear Regression: Models the relationship between input features and the target using a linear equation.
- Ridge Regression: Similar to Linear Regression but applies regularization to reduce excessively large coefficients or overfitting.
- Decision Tree Regressor: Predicts numerical values by learning decision rules and dividing the data into smaller groups.

In [68]:
# Cross validation
cv_reg = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results_reg = []

for name, model in models_reg.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_reg),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train_reg,
        y_train_reg,
        cv=cv_reg,
        scoring="neg_mean_absolute_error",
        return_train_score=False
    )

    mae_scores = -scores["test_score"]

    results_reg.append({
        "Model": name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

results_reg_df = pd.DataFrame(results_reg)

results_reg_df = results_reg_df.sort_values(
    by="Mean MAE"
)

display(results_reg_df)

,Model,Mean MAE,Std MAE
0,Linear Regression,2.034322,0.095611
1,Ridge Regression,2.034609,0.094634
2,Decision Tree Regressor,3.144527,0.340650


A lower mean MAE indicates better prediction accuracy. Lower std MAE means less variation.

Ridge regression has slightly lower Mean MAE and Std MAE.

In [69]:
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("model", Ridge())
])

param_grid_reg = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

grid_reg = GridSearchCV(
    ridge_pipeline,
    param_grid_reg,
    cv=cv_reg,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_reg.fit(X_train_reg, y_train_reg)

print("Best parameters:", grid_reg.best_params_)
print("Best CV MAE:", -grid_reg.best_score_)

Best parameters: {'model__alpha': 0.01}
Best CV MAE: 2.034325186507753


We use GridSearchCV to find the best regularization strength (alpha) for Ridge Regression. Five different alpha values are tested using 5-fold cross-validation, with MAE as the evaluation metric. GridSearchCV selects the alpha value that produces the lowest average cross-validation MAE. Mean MAE is slightly improved after tuning.

### Final Test Evaluation

In [72]:
best_reg = grid_reg.best_estimator_

y_pred_reg = best_reg.predict(X_test_reg)

test_mae = mean_absolute_error(y_test_reg, y_pred_reg)

test_rmse = mean_squared_error(
    y_test_reg,
    y_pred_reg
) ** 0.5

print("Final Test MAE:", test_mae)
print("Final Test RMSE:", test_rmse)

Final Test MAE: 2.1383190955117133
Final Test RMSE: 3.273845797669657


We select the best Ridge Regression pipeline using the results from GridSearchCV and evaluate it on the held-out testing set. We calculate MAE and RMSE to measure prediction errors.

MAE represents the average absolute difference between actual and predicted resolution times, while RMSE gives greater weight to larger prediction errors. The test set is used only for this final evaluation.

In [75]:

print(reg_df["resolution_time_hours"].describe())

Q1 = reg_df["resolution_time_hours"].quantile(0.25)
Q3 = reg_df["resolution_time_hours"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = reg_df[
    (reg_df["resolution_time_hours"] < lower_bound) |
    (reg_df["resolution_time_hours"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(outliers))
display(outliers.head())


count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64
Lower bound: -5.940557578774017
Upper bound: 16.917779901664556
Number of outliers: 10


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
19,R0019,4.0,14.0,High,Phone,38.541669,17.981195
49,R0049,5.0,10.0,High,Chat,46.570515,24.108510
51,R0051,7.0,3.0,Low,Phone,83.017767,23.929861
113,R0113,8.0,1.0,Critical,Chat,82.259644,18.267015
188,R0188,8.0,11.0,Medium,Phone,4.134498,26.135421


It uses the Interquartile Range (IQR) method to identify unusually high or low resolution times. It calculates Q1, Q3 and IQR, then determine the lower and upper bounds. Values outside these bounds are considered potential outliers.


# REPORT: 
# Classification and Regression

## 1. Introduction

The objective of this lab is to build and evaluate machine learning models for two tasks: predicting ticket escalation (classification) and predicting ticket resolution time (regression). We compare different models using cross-validation and tune selected hyperparameters.

## 2. Experiment A: Classification

### Problem and Preprocessing

The goal is to predict whether a support ticket will be escalated using the `escalated` target variable. The `ticket_id` column was excluded because it is only an identifier.

Numerical features were processed using median imputation and standardization. Categorical features were processed using most-frequent imputation and one-hot encoding.

We used a 75/25 stratified train-test split to maintain the class distribution in both sets.

### Model Comparison

We compared Logistic Regression, K-Nearest Neighbors, and Decision Tree using 5-fold StratifiedKFold cross-validation. F1-score was selected because the target classes are imbalanced and it considers both precision and recall.

| Model | Mean F1 | Standard Deviation |
|---|---:|---:|
| Decision Tree | 0.4014 | 0.1041 |
| Logistic Regression | 0.3822 | 0.0538 |
| KNN | 0.3094 | 0.0738 |

Decision Tree achieved the highest average F1-score, while Logistic Regression showed more consistent performance across folds.

### Hyperparameter Tuning and Test Evaluation

We selected Logistic Regression for tuning using GridSearchCV. We tested C values of 0.01, 0.1, 1, 10 and 100. C controls regularization strength, which helps reduce overfitting.

- Best C: **10**
- Best CV F1-score: **0.3949600999465296**

The final model was evaluated using a confusion matrix and classification report to examine its precision, recall, false positives and false negatives.

## 3. Experiment B: Regression

### Problem and Preprocessing

The goal is to predict `resolution_time_hours`. The `ticket_id` column was excluded.

Numerical and categorical features were processed using imputation, standardization, and one-hot encoding, respectively.

### Model Comparison

We compared Linear Regression, Ridge Regression, and Decision Tree Regression using 5-fold KFold cross-validation. MAE was selected because it measures the average absolute prediction error, where lower values indicate better performance.

| Model | Mean MAE | Standard Deviation |
|---|---:|---:|
| Linear Regression | 2.034322 | 0.095611 |
| Ridge Regression | 2.034609 | 0.094634 |
| Decision Tree Regressor | 3.144527 | 0.340650 |

### Hyperparameter Tuning and Test Evaluation

We tuned Ridge Regression using GridSearchCV with alpha values of 0.01, 0.1, 1, 10, and 100. Alpha controls the strength of regularization.

- Best alpha: **0.01**
- Best CV MAE: **2.034325186507753**
- Test MAE: **2.1383190955117133**
- Test RMSE: **3.273845797669657**

### Outlier Detection

We used the Interquartile Range (IQR) method to identify potential outliers in resolution time.

- Number of outliers: **10**
- Lower bound: **-5.940557578774017**
- Upper bound: **16.917779901664556**

Outliers can affect model performance. MAE treats absolute errors equally, while RMSE penalizes larger errors more heavily.

The lower IQR bound was negative. This is mathematically acceptable because the IQR method calculates statistical thresholds that do not necessarily represent physically possible values. Since resolution time cannot be negative, the negative lower bound does not indicate a meaningful outlier.

## 4. Leakage Prevention

Preprocessing was included inside pipelines so that imputers, scalers, and encoders were fitted only on the training portion of each fold. The test set was kept separate during model selection and used only for final evaluation.

## 5. Reflection Questions

**1. Why stratify the classification split?**

To maintain the proportion of each class in the training and test sets, especially because the target classes are imbalanced.

**2. Why must preprocessing be inside the pipeline?**

To prevent data leakage by ensuring preprocessing is learned only from the training data in each cross-validation fold.

**3. What does fold variability tell you?**

It shows how consistent a model's performance is across different folds. Lower standard deviation generally indicates more stable performance.

**4. Why should the test set be evaluated only once?**

Repeated evaluation can influence model selection and lead to overfitting to the test set. Using it once provides a more reliable final evaluation.

**5. What deployment constraints might change your choice?**

Prediction speed, computational resources, memory usage, interpretability, and maintenance requirements may influence the choice of a model.

## 6. Conclusion

In this lab, we compared classification and regression models, applied preprocessing pipelines, performed cross-validation, and tuned hyperparameters. We also examined outliers and evaluated the selected models on separate test sets. The experiments demonstrated the importance of proper validation and preventing data leakage.
